<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-05-bigquery-ml/lesson-5.1-bqml/notebooks/GCP_Capstone_5.1_BQML.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5.1 ML in SQL — CREATE MODEL for Regression, Classification & Clustering
**Netsetos GenAI Engineering — GCP Capstone**

Train ML models with pure SQL on your DocuMind document data.


## Setup


In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

def run_query(sql):
    """Run a BQ query and return results as DataFrame."""
    return client.query(sql).to_dataframe()

def run_ddl(sql):
    """Run a DDL/DML statement."""
    job = client.query(sql)
    job.result()
    print(f'Done: {job.num_dml_affected_rows or "model created"}')

print(f'Connected to {PROJECT_ID}')


## Cell 1: Create Dataset + Sample Data


In [ ]:
run_ddl(f'CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.ml_models`')
run_ddl(f'CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.rag_data`')

# Create sample document features table
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.document_features` AS
SELECT * FROM UNNEST([
  STRUCT('d1' AS doc_id, 'Q4 Report' AS title, 'pdf' AS content_type, 12 AS page_count, 2.5 AS file_size_mb, 24 AS chunk_count, 8500 AS total_word_count, 354.0 AS avg_chunk_size, FALSE AS has_tables, TRUE AS has_images, 0.018 AS processing_cost_usd, 'research_paper' AS document_type),
  STRUCT('d2','Invoice 2024-001','pdf',2,0.3,4,800,200.0,TRUE,FALSE,0.003,'invoice'),
  STRUCT('d3','NDA Agreement','pdf',8,1.1,16,4200,262.0,FALSE,FALSE,0.012,'legal'),
  STRUCT('d4','KYC Form','pdf',1,0.2,2,400,200.0,TRUE,FALSE,0.002,'form'),
  STRUCT('d5','ML Survey 2025','pdf',45,8.2,90,32000,355.0,TRUE,TRUE,0.068,'research_paper'),
  STRUCT('d6','Receipt Mar','pdf',1,0.1,2,250,125.0,TRUE,FALSE,0.002,'invoice'),
  STRUCT('d7','License Agreement','pdf',15,2.8,30,12000,400.0,FALSE,FALSE,0.023,'legal'),
  STRUCT('d8','Employee Onboarding','pdf',3,0.5,6,1800,300.0,TRUE,FALSE,0.005,'form'),
  STRUCT('d9','RAG Architecture','pdf',22,4.1,44,16000,363.0,TRUE,TRUE,0.033,'research_paper'),
  STRUCT('d10','GST Invoice','pdf',1,0.15,2,350,175.0,TRUE,FALSE,0.002,'invoice'),
  STRUCT('d11','Partnership Deed','pdf',20,3.5,40,15000,375.0,FALSE,FALSE,0.030,'legal'),
  STRUCT('d12','Leave Application','pdf',1,0.1,2,300,150.0,TRUE,FALSE,0.002,'form')
])
''')
print('Sample data created')
run_query(f'SELECT doc_id, title, document_type, processing_cost_usd FROM `{PROJECT_ID}.rag_data.document_features`')


## Cell 2: Linear Regression — Predict Processing Cost


In [ ]:
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.cost_predictor`
OPTIONS (
  model_type = 'LINEAR_REG',
  input_label_cols = ['processing_cost_usd'],
  max_iterations = 20,
  early_stop = TRUE,
  l2_reg = 0.1,
  enable_global_explain = TRUE
) AS
SELECT
  page_count, chunk_count, total_word_count,
  file_size_mb, content_type, processing_cost_usd
FROM `{PROJECT_ID}.rag_data.document_features`
WHERE processing_cost_usd IS NOT NULL
''')
print('Cost predictor trained')


## Cell 3: Evaluate + Predict


In [ ]:
# Evaluate
print('=== Regression Metrics ===')
print(run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.cost_predictor`)'))

# Predict
print('\n=== Predictions ===')
print(run_query(f'''
SELECT doc_id, title, 
  ROUND(predicted_processing_cost_usd, 4) AS predicted_cost
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.ml_models.cost_predictor`,
  (SELECT * FROM `{PROJECT_ID}.rag_data.document_features`))
'''))

# Feature importance
print('\n=== Feature Importance ===')
print(run_query(f'SELECT * FROM ML.GLOBAL_EXPLAIN(MODEL `{PROJECT_ID}.ml_models.cost_predictor`)'))


## Cell 4: Logistic Regression — Classify Document Types


In [ ]:
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.doc_classifier`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['document_type'],
  auto_class_weights = TRUE,
  max_iterations = 20,
  enable_global_explain = TRUE
) AS
SELECT
  page_count, total_word_count, avg_chunk_size,
  chunk_count, file_size_mb, has_tables, has_images,
  document_type
FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('Document classifier trained')

# Evaluate
print('\n=== Classification Metrics ===')
print(run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.doc_classifier`)'))

# Confusion matrix
print('\n=== Confusion Matrix ===')
print(run_query(f'SELECT * FROM ML.CONFUSION_MATRIX(MODEL `{PROJECT_ID}.ml_models.doc_classifier`)'))


## Cell 5: K-Means Clustering (on synthetic embeddings)


In [ ]:
# Create synthetic embedding data (in production, use real embeddings from Lesson 2.2)
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.chunk_embeddings` AS
SELECT
  CONCAT('chunk_', CAST(n AS STRING)) AS chunk_id,
  CONCAT('doc_', CAST(MOD(n, 5) + 1 AS STRING)) AS doc_id,
  GENERATE_ARRAY(RAND(), RAND() + 0.01, (RAND() + 0.01 - RAND()) / 9) AS embedding
FROM UNNEST(GENERATE_ARRAY(1, 50)) AS n
''')

# Train K-Means
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.topic_clusters`
OPTIONS (
  model_type = 'KMEANS',
  num_clusters = 5,
  kmeans_init_method = 'KMEANS++',
  distance_type = 'COSINE',
  standardize_features = FALSE,
  max_iterations = 50
) AS
SELECT embedding
FROM `{PROJECT_ID}.rag_data.chunk_embeddings`
WHERE embedding IS NOT NULL
''')
print('Topic clusters trained')

# Evaluate
print('\n=== Clustering Metrics ===')
print(run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.topic_clusters`)'))

# Predict cluster assignments
print('\n=== Cluster Assignments ===')
print(run_query(f'''
SELECT chunk_id, doc_id, CENTROID_ID AS topic_cluster
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.ml_models.topic_clusters`,
  (SELECT * FROM `{PROJECT_ID}.rag_data.chunk_embeddings`))
ORDER BY topic_cluster
LIMIT 20
'''))


## Cell 6: Boosted Tree Upgrade with TRANSFORM


In [ ]:
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.doc_classifier_boosted`
TRANSFORM (
  page_count,
  ML.STANDARD_SCALER(total_word_count) OVER() AS scaled_words,
  ML.QUANTILE_BUCKETIZE(file_size_mb, 5) OVER() AS size_bucket,
  chunk_count, avg_chunk_size, has_tables, has_images,
  document_type
)
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['document_type'],
  max_tree_depth = 4,
  max_iterations = 30,
  learn_rate = 0.1,
  subsample = 0.8,
  early_stop = TRUE,
  enable_global_explain = TRUE
) AS
SELECT * FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('Boosted tree classifier trained')

# Compare with logistic regression
print('\n=== Logistic Reg Metrics ===')
print(run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.doc_classifier`)'))
print('\n=== Boosted Tree Metrics ===')
print(run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.doc_classifier_boosted`)'))


## Cell 7: Training Progress + Explainability


In [ ]:
# Training loss curve
print('=== Training Progress ===')
print(run_query(f'''
SELECT iteration, training_loss, eval_loss, duration_ms
FROM ML.TRAINING_INFO(MODEL `{PROJECT_ID}.ml_models.cost_predictor`)
ORDER BY iteration
'''))

# Explain individual predictions
print('\n=== Explained Predictions ===')
print(run_query(f'''
SELECT *
FROM ML.EXPLAIN_PREDICT(
  MODEL `{PROJECT_ID}.ml_models.cost_predictor`,
  (SELECT * FROM `{PROJECT_ID}.rag_data.document_features` LIMIT 3),
  STRUCT(3 AS top_k_features))
'''))


## ✅ Lesson 5.1 Complete!

- ✅ CREATE MODEL for LINEAR_REG, LOGISTIC_REG, KMEANS
- ✅ ML.EVALUATE with r2_score, precision, recall, davies_bouldin
- ✅ ML.PREDICT for inference on new data
- ✅ ML.EXPLAIN_PREDICT with Shapley values
- ✅ ML.CONFUSION_MATRIX + ML.GLOBAL_EXPLAIN
- ✅ TRANSFORM clause for preprocessing
- ✅ BOOSTED_TREE_CLASSIFIER upgrade
- ✅ 3-model DocuMind integration pattern

**Next: Lesson 5.2 — ML.GENERATE_TEXT: Gemini in SQL**
